In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
!pip install wandb

In [0]:
import os
import sys
import warnings
import logging

# 1. Silence Databricks & Py4J Thread noise
logging.getLogger("ThreadMonitor").setLevel(logging.ERROR)
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)

# 2. Silence TensorFlow/XLA C++ warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

sys.path.append(os.path.abspath('..'))
wandb_api_key = dbutils.secrets.get(scope='haroon-scope', key='WANDA_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
warnings.filterwarnings('ignore')

from trainer import Trainer, TrainingArguments
from bacp import BaCPTrainingArguments, BaCPTrainer
from utils import set_seed

In [0]:
from pruning_factory import *
from model_factory import *
from utils import *
from weight_sharing import *
import matplotlib.pyplot as plt


#### Cyclic Sparsity Function

In [0]:
d_max = 1 - 0.995
d_min = 1 - 0.999
current_idx = 0
Tc = 150
T_end = 250
n = 2 # no. of cycles
vals = []
for i in range(current_idx, T_end):
    if i <= 50:
        norm_t = i / 50
        d_target = d_min + (d_max - d_min) * 0.5 * (1 - math.cos((2 * math.pi * norm_t)))
        vals.append(d_target)
    elif  0 < i - 50 <= 100:
        norm_t = (i - 50) / 100
        d_target = d_min + (d_max - d_min) * 0.5 * (1 - math.cos(2 * math.pi * norm_t))
        vals.append(d_target)
    else:
        vals.append(d_min)

plt.plot(range(T_end), vals)


In [0]:
model = ClassificationAndEncoderNetwork('resnet34', num_classes=10, dyrelu_en=True)
load_weights(model, '/dbfs/research/bacp/resnet34/cifar10/20260122/resnet34_cifar10_dense_baseline_relu_20260122_195252.pt')
apply_weight_sharing_resnet(model)

pruning_type = 'east'
pruner = get_pruner(
    method_name=pruning_type,
    model=model,
    epochs=20,
    sparsity=0.9,
    scheduler_type='cyclic',
    delta_T=100,
    total_steps=2000,
    alpha=0.3 if pruning_type in ('rigl', 'east') else None,
)

In [0]:
d_max = 1 - 0.995
d_min = 1 - 0.999
current_idx = 0
Tc = 100
T_end = 150
n = 2 # no. of cycles
vals = []
for i in range(current_idx, T_end):
    if i <= Tc:
        norm_t = i / Tc
        d_target = d_min + (d_max - d_min) * 0.5 * (1 - math.cos((2 * math.pi * norm_t) * n))
        vals.append(d_target)
    else:
        vals.append(d_min)

plt.plot(range(T_end), vals)


### RigL Tests

--trained_weights /dbfs/research/bacp/resnet34/cifar10/20260122/resnet34_cifar10_dense_baseline_relu_20260122_195252.pt \


In [0]:
!python ../scripts/pruning_script.py \
    --model_name resnet34 \
    --model_type cv \
    --dataset_name cifar10 \
    --num_classes 10 \
    --log_to_wandb --databricks_env \
    --trained_weights None \
    --experiment_type "rigl_test" \
    --pruning_type rigl --target_sparsity 0.9995 --sparsity_scheduler f_decay \
    --epochs 500 --learning_rate 0.1 --scheduler_type cosine --delta_T 100

In [0]:
from pruning_factory import check_sparsity_distribution
from model_factory import ClassificationAndEncoderNetwork
from utils import load_weights
model = ClassificationAndEncoderNetwork('resnet34', num_classes=10)
load_weights(model, "/dbfs/research/bacp/resnet34/cifar10/20260222/resnet34_cifar10_rigl_0.99_rigl_test_20260222_072953.pt")
check_sparsity_distribution(model)